# TESSNet

In [ ]:
!pip install lightkurve torch torchvision matplotlib numpy pandas scikit-learn astroquery
!pip install "fsspec==2025.3.0" "fsspec[http]==2025.3.0" "s3fs==2025.3.0" "gcsfs==2025.3.0"

### Cell 1: Installations and Imports

Run this cell to install the necessary astronomy and deep learning libraries.

In [ ]:
# Install required packages (uncomment if running in Colab/Kaggle)
# !pip install lightkurve torch torchvision matplotlib numpy pandas scikit-learn

import os
import lightkurve as lk
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import warnings

# Suppress lightkurve warnings for cleaner output
warnings.filterwarnings('ignore')

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### Cell 2: Data Processing & Image Generation Functions

This cell defines the core logic to download a light curve, clean it, find the period, phase-fold it, and save it as an image for the CNN.

In [ ]:
def process_and_save_lightcurve(tic_id, save_dir, class_label):
    """
    Downloads TESS data, processes it, and saves a 2D image of the folded light curve.
    """
    try:
        # 1. Download High-Cadence Data (using the fastest available sector)
        search_result = lk.search_lightcurve(f"TIC {tic_id}", mission="TESS", author="SPOC")
        if len(search_result) == 0:
            return None

        lc = search_result[0].download()

        # 2. Preprocess: Remove NaNs and flatten the light curve
        lc = lc.remove_nans().remove_outliers(sigma_upper=3, sigma_lower=20)
        flat_lc, trend = lc.flatten(window_length=101, return_trend=True)

        # 3. Find the dominant period using Box Least Squares (BLS)
        periodogram = flat_lc.to_periodogram(method='bls', period=np.arange(0.5, 20, 0.01))
        best_period = periodogram.period_at_max_power
        best_t0 = periodogram.transit_time_at_max_power
        best_duration = periodogram.duration_at_max_power

        # 4. Phase-fold the light curve
        folded_lc = flat_lc.fold(period=best_period, epoch_time=best_t0)

        # 5. Generate and save the 2D image
        os.makedirs(os.path.join(save_dir, class_label), exist_ok=True)
        filename = os.path.join(save_dir, class_label, f"TIC_{tic_id}.png")

        # Plotting settings optimized for CNNs (no axes, no labels)
        fig, ax = plt.subplots(figsize=(3, 3), dpi=85) # Roughly 255x255 pixels
        ax.scatter(folded_lc.time.value, folded_lc.flux.value, s=1, c='black', alpha=0.5)
        ax.axis('off') # Hide axes

        plt.margins(0,0)
        fig.savefig(filename, bbox_inches='tight', pad_inches=0, facecolor='white')
        plt.close(fig)

        # Return parameters for later evaluation
        return {
            'tic_id': tic_id,
            'period': best_period.value,
            'duration': best_duration.value,
            'depth': periodogram.depth_at_max_power.value
        }

    except Exception as e:
        print(f"Failed to process TIC {tic_id}: {e}")
        return None

### Cell 3: Generating the Dataset

This cell simulates processing your curated dataset. You will need to replace the `curated_targets` dictionary with your actual list of TIC IDs.

In [ ]:
from astroquery.mast import Catalogs
import random
import os
import numpy as np

print("Querying MAST for TESS Input Catalog targets...")

# 1. Fetching a batch of valid TIC IDs via Astroquery
# Query a 3-degree region of the sky. We remove the Tmag parameter from the API call.
catalog_data = Catalogs.query_region("60 -60", radius="3 deg", catalog="TIC")

# 2. Filter locally
# The API returns an AstroPy Table. We filter out targets with no magnitude data
# and then select moderately bright stars (Tmag < 12) so we don't get pure noise.
valid_mask = (~catalog_data['Tmag'].mask) & (catalog_data['Tmag'] < 12)
filtered_data = catalog_data[valid_mask]
all_sector_tics = filtered_data['ID'].tolist()

print(f"Successfully loaded and filtered {len(all_sector_tics)} bright targets from the TIC.")

# 3. Integrating the "Curated Dataset"
curated_targets = {
    'transits': [27491137, 25155310, 261136679, 307210830],
    'eclipsing_binaries': [11446443, 220430635, 278683839],
    # Randomly sample 200 background stars to act as our noise/null class
    'noise': random.sample(all_sector_tics, min(200, len(all_sector_tics)))
}

DATA_DIR = "./tess_images"
extracted_parameters = []

print("Generating dataset images... (This may take several minutes)")
for class_label, tic_list in curated_targets.items():
    print(f"Processing class: {class_label} ({len(tic_list)} targets)")
    for tic in tic_list:
        # process_and_save_lightcurve is defined in your Cell 2
        params = process_and_save_lightcurve(tic, DATA_DIR, class_label)
        if params:
            params['label'] = class_label
            extracted_parameters.append(params)

print(f"Image generation complete! {len(extracted_parameters)} targets successfully processed.")

### Cell 4: PyTorch Data Loaders

Prepares the images for the neural network using `torchvision`.

In [ ]:
# Define image transformations (resize and normalize for ResNet)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), # Data augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load dataset from the generated directory
dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)

# Split into training (80%) and validation (20%)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

classes = dataset.classes
print(f"Classes detected: {classes}")

### Cell 5: Model Definition

We use a pre-trained ResNet18 and modify the final classification layer.

In [ ]:
# Load pre-trained ResNet18
model = models.resnet18(pretrained=True)

# Freeze early layers to speed up training (optional)
for param in model.parameters():
    param.requires_grad = False

# Modify the final layer for our specific number of classes
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(classes)) # Output = number of classes

model = model.to(device)

# Define Loss function and Optimizer
criterion = nn.CrossEntropyLoss()
# Only optimize the final layer we just created
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

### Cell 6: Training Loop

Executes the training and validation over multiple epochs.

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # Training phase
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    # Validation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total if val_total > 0 else 0

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss/max(1, len(val_loader)):.4f} | Val Acc: {val_acc:.2f}%")

### Cell 7: Inference & Parameter Reporting

Demonstrates how to pass a single image through the trained model to get confidence levels, and pair it with the physical parameters calculated in Cell 2.

In [ ]:
def predict_and_evaluate(image_path, model, class_names):
    from PIL import Image

    # Load and transform the image
    image = Image.open(image_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(device)

    # Run inference
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        probabilities = torch.nn.functional.softmax(output[0], dim=0)
        confidence, predicted_idx = torch.max(probabilities, 0)

    predicted_class = class_names[predicted_idx.item()]
    confidence_pct = confidence.item() * 100

    return predicted_class, confidence_pct

# --- Example Usage ---
# Assuming we want to evaluate the first item we processed in Cell 3
if extracted_parameters:
    sample = extracted_parameters[0]
    sample_img_path = os.path.join(DATA_DIR, sample['label'], f"TIC_{sample['tic_id']}.png")

    if os.path.exists(sample_img_path):
        pred_class, conf = predict_and_evaluate(sample_img_path, model, classes)

        print(f"--- Results for TIC {sample['tic_id']} ---")
        print(f"AI Classification: {pred_class} (Confidence: {conf:.2f}%)")
        print(f"Calculated Parameters (from Lightkurve BLS):")
        print(f"- Orbital Period:  {sample['period']:.4f} days")
        print(f"- Transit Duration: {sample['duration']:.4f} days")
        print(f"- Transit Depth:    {sample['depth']:.6f} fractional flux")

### Cell 8: Downloading the Model

This saves the state_dict (the weights and biases your model learned during those training epochs). It is the recommended way to save PyTorch models because it keeps the file size small.

In [ ]:
# 1. Save the model's learned weights (state_dict) to the Colab environment
model_save_path = "tess_resnet18_model.pth"
torch.save(model.state_dict(), model_save_path)
print(f"Model weights successfully saved locally to {model_save_path}")

# 2. Trigger the browser download to save it to your local machine
from google.colab import files
files.download(model_save_path)